# Lab L0105: Context windows and cost

```yaml
title: "Lab L0105: Context windows and cost"
keywords: context window, headroom, cost, pricing, input output asymmetry, conversation history, lab
estimated duration: 35
```

> **Lesson:** L01. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L01/objectives.md).
> Solutions: `L0105_lab_solutions.ipynb`. This lab is pure arithmetic — no API key needed.

**After this lab you will be able to:**
- Estimate whether a call fits the context window and how much headroom remains.
- Compute the cost of a single call from token counts.
- Explain the input/output price asymmetry and the conversation-history staircase.

## Setup

Anchor model: **Claude Sonnet 4.6**, 200,000-token window. Rates below are **illustrative** — always
confirm current pricing on Anthropic's pricing page.

In [1]:
WINDOW = 200_000  # tokens
INPUT_USD_PER_MTOK = 3.00  # illustrative only
OUTPUT_USD_PER_MTOK = 15.00  # illustrative only; output is the expensive direction
print("setup ready")

setup ready


### Problem 1 — Headroom

Implement `headroom`: given the token counts of the system message, conversation history, and
current input, plus the tokens you want to reserve for the output, return how many tokens are left
in the window. A negative number means it won't fit.

In [2]:
def headroom(system: int, history: int, current_input: int, reserved_output: int) -> int:
    return WINDOW - (system + history + current_input + reserved_output)


print("tiny call   :", headroom(20, 0, 15, 500))
print("long history:", headroom(20, 180_000, 15, 500))
print("overflow    :", headroom(20, 205_000, 15, 500))  # expect negative

tiny call   : 199465
long history: 19465
overflow    : -5535


### Problem 2 — Match the failure mode

For each scenario, name which failure mode applies: **hard rejection**, **silent truncation**, or
**quality degradation**.

1. You send 250k tokens to a 200k-window model and the API returns an error.
2. A chat framework keeps the conversation "working" but the model no longer recalls something you
   said 40 turns ago.
3. A call fits the window, but answers about details buried in the middle of a very long document
   get vaguer.

> _TODO: write `1 -> ...`, `2 -> ...`, `3 -> ...` here._

### Problem 3 — Cost of one call

Implement `call_cost_usd` using the per-**million**-token rates, then compute the cost of a call
with 1,500 input tokens and 300 output tokens.

In [3]:
def call_cost_usd(input_tokens: int, output_tokens: int) -> float:
    return (
        input_tokens / 1_000_000 * INPUT_USD_PER_MTOK
        + output_tokens / 1_000_000 * OUTPUT_USD_PER_MTOK
    )


print(f"${call_cost_usd(1500, 300):.5f}")

$0.00900


### Problem 4 — The input/output asymmetry

Compute the cost of (a) 2,000 input + 50 output tokens and (b) 50 input + 2,000 output tokens.
Which is more expensive, and what does that imply for prompt design?

In [4]:
long_in_short_out = call_cost_usd(2000, 50)
short_in_long_out = call_cost_usd(50, 2000)
print(f"long input,  short answer: ${long_in_short_out:.5f}")
print(f"short input, long answer:  ${short_in_long_out:.5f}")
# short-input/long-output is pricier because output tokens cost ~5x input here, so paying
# the model to WRITE a lot is more expensive than paying it to READ a lot.

long input,  short answer: $0.00675
short input, long answer:  $0.03015


### Problem 5 — The conversation-history staircase

A 5-turn chat adds ~200 input tokens and produces ~60 output tokens per turn, and the whole history
is re-sent each turn. Print the cumulative input tokens and per-turn cost, then the session total.

In [5]:
PER_TURN_INPUT = 200
PER_TURN_OUTPUT = 60
cumulative = 0
total = 0.0
for t in range(1, 6):
    cumulative += PER_TURN_INPUT
    cost = call_cost_usd(cumulative, PER_TURN_OUTPUT)
    total += cost
    print(f"turn {t}: input re-sent ~{cumulative:4} tokens -> ${cost:.5f}")
print(f"session total: ${total:.5f}")

turn 1: input re-sent ~ 200 tokens -> $0.00150
turn 2: input re-sent ~ 400 tokens -> $0.00210
turn 3: input re-sent ~ 600 tokens -> $0.00270
turn 4: input re-sent ~ 800 tokens -> $0.00330
turn 5: input re-sent ~1000 tokens -> $0.00390
session total: $0.01350


### Problem 6 (stretch) — Order-of-magnitude agent budget

A single agent step costs about `call_cost_usd(2000, 300)`. Estimate the cost of a 10-step run, then
of running that 10-step agent 100 times during development. Is the dev-iteration number "too small to
matter"?

In [6]:
step = call_cost_usd(2000, 300)
run_10 = step * 10
dev_100 = run_10 * 100
print(f"one step ~${step:.5f}; 10-step run ~${run_10:.4f}; 100 dev runs ~${dev_100:.2f}")

one step ~$0.01050; 10-step run ~$0.1050; 100 dev runs ~$10.50


## Self-check

No automatic grader — compare against `L0105_lab_solutions.ipynb`. You're done when your headroom
sign flips correctly on overflow and your staircase total matches.